# P12-P13 · ICF y distribución de la frecuencia programada

> **¿Qué parte del ICF se puede calcular con GTFS y cómo se distribuye la frecuencia programada?**

frequencies.txt, trips.txt y routes.txt del GTFS vigente, más la definición oficial del ICF.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/frecuencia_programada/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-red-movilidad


## 1 · El problema

El ICF oficial parece una razón simple, pero su numerador y denominador son mediciones
GPS. El GTFS solo contiene el programa. Confundir ambos convierte una promesa de oferta
en una medición de cumplimiento inexistente. El mismo archivo sí permite contestar P13:
cómo se distribuyen los intervalos programados y qué servicios tienen mayor espera.


## 2 · La solución que se va a aplicar

Se validan las ventanas, se calculan sus salidas con fin exclusivo y se construye una
matriz de alcance del ICF. La distribución se resume con percentiles y tres unidades de
análisis. La punta mañana se filtra por intersección temporal, no por hora de inicio.


## Implementaciones comparables

La ruta principal usa Pandas. Polars y R recalculan el indicador central desde el contrato de salida versionado y generan el mismo gráfico.


### Polars + Matplotlib

```python
"""Contraste Polars y visualización Matplotlib del caso frecuencia_programada."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "frecuencia_programada" / "resultados" / "distribucion.csv"


def analizar() -> pl.DataFrame:
    datos = pl.read_csv(SOURCE)
    return datos.select(["estadistico", "intervalo_min"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    x = resultado["estadistico"].to_list()
    y = resultado["intervalo_min"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([str(value) for value in x], y, color="#11253B")
    ax.set_xlabel("Estadístico")
    ax.set_ylabel("Intervalo entre salidas (minutos)")
    ax.set_ylim(bottom=0)
    ax.set_title("La distribución revela más que un promedio")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P12/P13 · Período: publicación GTFS analizada",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--grafico", type=Path)
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/frecuencia_programada/cuaderno/solucion_polars.py`.


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Contraste R + ggplot2 del caso frecuencia_programada.
script_arg <- grep("^--file=", commandArgs(trailingOnly = FALSE), value = TRUE)
script_path <- if (length(script_arg)) sub("^--file=", "", script_arg[1]) else "cuaderno/solucion.R"
root <- normalizePath(file.path(dirname(script_path), "../../.."), mustWork = TRUE)
source_csv <- file.path(root, "cases", "frecuencia_programada", "resultados", "distribucion.csv")

datos <- read.csv(source_csv, check.names = FALSE, fileEncoding = "UTF-8")
resultado <- datos[, c("estadistico", "intervalo_min")]
print(resultado, row.names = FALSE)

args <- commandArgs(trailingOnly = TRUE)
plot_pos <- match("--grafico", args)
if (!is.na(plot_pos)) {
  if (plot_pos == length(args)) stop("--grafico necesita una ruta de salida")
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  destino <- args[plot_pos + 1]
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["estadistico"]], y = .data[["intervalo_min"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::labs(
      title = "La distribución revela más que un promedio",
      x = "Estadístico", y = "Intervalo entre salidas (minutos)",
      caption = "Fuente: Resultados reproducibles del caso P12/P13 · Período: publicación GTFS analizada"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(destino, grafico, width = 9, height = 5, dpi = 150)
  cat(destino, "\n")
}
```

Ejecutar con `Rscript cases/frecuencia_programada/cuaderno/solucion.R`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.frecuencia_programada import solution as frecuencia

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Leer solo tres tablas


El ICF programado no necesita el archivo de 1,1 millones de horarios.

> ⚠️ Cargar por costumbre la tabla grande duplica memoria sin aportar una columna al cálculo.


In [ ]:
tablas = frecuencia.cargar()
{k: v.shape for k, v in tablas.items()}


### Paso 2 · Validar las ventanas


Inicio, fin e intervalo se tipan antes de resumir.

> ⚠️ Una ventana inválida no se descarta sin contarla.


In [ ]:
ventanas = frecuencia.ventanas_programadas(tablas)
ventanas[['valida','salidas_programadas']].agg(['count','sum'])


### Paso 3 · Separar programa de observación


La matriz impide presentar el programa como ICF.

> ⚠️ 142 366 salidas programadas no son 142 366 mediciones GPS.


In [ ]:
resultados = frecuencia.analizar()
resultados['alcance_icf']


### Paso 4 · Conservar el ICF como no calculable


La ausencia de GPS es un resultado de alcance, no un error de ejecución.

> ⚠️ Rellenar con cero produciría un ICF de 0 % que los datos jamás observaron.


In [ ]:
resultados['alcance_icf'].query("calculable_con_gtfs == 'NO'")


### Paso 5 · Describir la distribución


Percentiles muestran la asimetría entre 3,10 y 330 minutos.

**En Excel:** `=PERCENTIL.INC(rango_intervalo_min;0,5)`

Debe dar: **15 minutos**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ La media de 21,17 minutos no representa la ventana típica de 15.


In [ ]:
resultados['distribucion']


### Paso 6 · Cambiar la unidad de análisis


Ventana, servicio y salida responden preguntas distintas.

> ⚠️ Promediar servicios sin declarar peso da el mismo voto a uno diario y a uno intensivo.


In [ ]:
resultados['ponderaciones']


### Paso 7 · Filtrar la punta por intersección


Entra toda ventana que toca 08:00–09:00.

> ⚠️ Filtrar solo `start_time` pierde ventanas que comenzaron antes de las 08:00.


In [ ]:
resultados['punta_manana'].head(15)


### Paso 8 · Publicar una conclusión proporcional


Se distingue frecuencia larga de incumplimiento.

> ⚠️ Un intervalo programado largo puede ser diseño de servicio; no demuestra falla operacional.


In [ ]:
resultados['ventanas_invalidas']


## 4 · El resultado

El GTFS permite construir 142 366 salidas programadas y describir sus intervalos. No contiene mediciones GPS: el ICF oficial permanece no calculable.

## 5 · Qué queda aprendido

1. Programa no es operación observada.
2. La unidad de análisis cambia el intervalo típico.
3. Las ventanas se filtran por intersección temporal.
4. Un vacío metodológico no se rellena con cero.
